In [1]:
import os
import sys

project_root = os.path.abspath("..")   # if the notebook is inside notebooks/
sys.path.insert(0, project_root)
print(project_root)

c:\Projects\Service Event Framework\Improvement\code


In [2]:
import os
import numpy as np
import pandas as pd
from utils.dbconnection import engine
from pathlib import Path
from utils.config import config

2026-10-08 10:48:20,932 - Logger: Log file: c:\Projects\Service Event Framework\Improvement\code\logs\SEF_Improvement_20261008_104820.log
2026-10-08 10:48:20,958 - SQLAlchemy: Database connection configured successfully!


In [87]:
parent = os.path.join(config.locations.inputs, "material_as_parent_child.csv")
df_parent = pd.read_csv(parent)
df_parent.columns = ['parent_material', 'child_material']
order_count_path = os.path.join(config.locations.inputs, "Material_Order_Counts_v2.xlsx")
df_order_counts = pd.read_excel(order_count_path)

In [88]:
df_parent.head(2), df_order_counts.head(2)

(  parent_material child_material
 0          100223         109283
 1          101110         153184,
    Material                    MaterialText  order_counts
 0    131496            CON SP S G1(M)XG1(F)           846
 1  10212870  GREASE COLLECTING CAN W. COVER          2795)

Availability of common Materials

In [126]:
df_parent['parent_material'].nunique(), df_parent['child_material'].nunique()

(2532, 7741)

In [127]:
child_pres = df_order_counts.merge(df_parent, left_on='Material', right_on='child_material', how='inner')
par_pres = df_order_counts.merge(df_parent, left_on='Material', right_on='parent_material', how='inner')
child_pres.shape, par_pres.shape

((1284, 5), (221, 5))

In [89]:
xlsx_path = os.path.join(config.locations.outputs, "service_event_analysis.xlsx")
df = pd.read_excel(xlsx_path, sheet_name="event_material")
df.head()

,DServiceEventID,Serviceeventcode,ServiceEventName,AND_OR,Priority,confidence,Itemnumber,MaterialName,MaterialGroup,MaterialGroupDescription
0,1,BladeBearingReplace,Blade Bearing Replacement,OR,9.6,10,107033,BLADE BEARING ø54,31,"Bearings, Blade TRAC"
1,1,BladeBearingReplace,Blade Bearing Replacement,OR,9.6,10,760306,BLADE BEARING MODULE,327,Nacelle modules and
2,1,BladeBearingReplace,Blade Bearing Replacement,OR,9.6,10,76221907,BLADE BEARING MODULE,30,Bearings
3,1,BladeBearingReplace,Blade Bearing Replacement,OR,9.6,10,76221971,BLADE BEARING MODULE,327,Nacelle modules and
4,4,GenBearReplace,Generator Bearing Replacement,OR,7.8,10,103566,"BEARING,BALL,6344 MC3,220 mm",30,Bearings


In [90]:
df['Serviceeventcode'].nunique()

486

In [91]:
events = ",".join("'" + df['Serviceeventcode'].unique().astype(str) + "'")
events

"'BladeBearingReplace','GenBearReplace','MainBearingReplace','PitchCylinderReplace','PropValveReplace','YawGearReplace','GearIMS','GearHSS','GearPlanetaryStage','GearPT100SensorReplace','GearPitchTubeLeakRepair','GenFanReplace','CACPumpReplace','CACLiquidConsumption','HydOilConsumption','HydMotorReplace','HydPumpReplace','GearOilConsumption','TrafOilConsumption','YawMotorReplace','ServiceLiftMotorReplace','CoolingSystemFanMotorReplace','GearLubeMotorReplace','CoolerTopReplace','SpinnerNoseConeReplace','WindvaneSensorReplace','CACPressSensorReplace','CCISensorReplace','CACSensorReplace','PPRSensorReplace','CCICBReplace','LPCCBReplace','PPRCBReplace','YAWCBReplace','CCIKitsetConsumption','NLEKitsetConsumption','PTRKitsetConsumption','YAWKitsetConsumption','GenBrushReplace','GenRotUnitAdaptorReplace','GbxRepairAdhesiveConsume','PPRCimUpgrCBReplace','PPRContactorReplace','PPRSkiipReplace','BBGreaseConsumption','GenGreaseConsumption','HydLubConsumption','MBAGreaseConsumption','TLFLubConsump

In [92]:
df['Itemnumber'].nunique()

8129

Frame the query to be run in Snowflake

In [93]:
df.dropna(inplace=True)

In [94]:
merged_ids = ",".join("'" + df['Itemnumber'].astype(str) + "'")
merged_ids

"'107033','760306','76221907','76221971','103566','105203','105243','105244','107045','107062','107064','107075','107080','107081','107085','107086','107087','107088','107089','107090','107091','107092','107146','107152','107155','107162','107163','107164','107166','107170','107173','107174','107178','107247','107317','151902','151903','151906','151907','29001915','29001916','29013250','29018808','29018809','29055811','29078688','29085546','29088145','29091280','29092454','29092464','29094865','29098041','29101168','29106634','29108850','29142587','29151345','29185782','29185818','29195706','34721','34729','34736','34744','60036055','60037816','60042600','60046041','60050981','60068860','60078227','60079086','60079090','60079092','60079328','60091771','60113590','60113591','60114065','60120942','70000147','70590033','70590034','29155641','29195008','108613','108614','110912','110915','110915REN','29058278','29060554','29060555','29060556','29080628','29080630','29084354','29084355','29

In [95]:
snflk_query = '''select distinct
part_number as parent, part_name as parent_name, 
parent as material, parent_description as material_name,
component_number as child_number, 
TRIM(REPLACE(structure_name, '|', '')) as child_name
from OFFSHORE_READINESS.DATAMART.DIM_SAP_BOM_EXPLOSION
where 1=1
and year(bom_valid_to)=9999
and parent is not null
and parent != part_number
and parent in ({})
'''.format(merged_ids)
print(snflk_query)

select distinct
part_number as parent, part_name as parent_name, 
parent as material, parent_description as material_name,
component_number as child_number, 
TRIM(REPLACE(structure_name, '|', '')) as child_name
from OFFSHORE_READINESS.DATAMART.DIM_SAP_BOM_EXPLOSION
where 1=1
and year(bom_valid_to)=9999
and parent is not null
and parent != part_number
and parent in ('107033','760306','76221907','76221971','103566','105203','105243','105244','107045','107062','107064','107075','107080','107081','107085','107086','107087','107088','107089','107090','107091','107092','107146','107152','107155','107162','107163','107164','107166','107170','107173','107174','107178','107247','107317','151902','151903','151906','151907','29001915','29001916','29013250','29018808','29018809','29055811','29078688','29085546','29088145','29091280','29092454','29092464','29094865','29098041','29101168','29106634','29108850','29142587','29151345','29185782','29185818','29195706','34721','34729','34736','34744','60

Execution flow for one material

In [96]:
df['Itemnumber'].values[0]

'107033'

In [97]:
parents_df = pd.DataFrame()
for itemnumber in df['Itemnumber'].values:
    parents_df = pd.concat([parents_df, df_parent.loc[df_parent['parent_material']==itemnumber]])
parents_df.head()

,parent_material,child_material
12543,760306,103850
12544,760306,103852
12545,760306,105717
12546,760306,107028
12547,760306,108088


In [98]:
p1 = parents_df.loc[parents_df['parent_material']=='760306']
p1.head(2)

,parent_material,child_material
12543,760306,103850
12544,760306,103852


In [101]:
df_order_counts.columns, df_order_counts.shape

(Index(['Material', 'MaterialText', 'order_counts'], dtype='str'), (67396, 3))

In [102]:
type(p1['parent_material'].values[0]), type(df_order_counts['Material'].values[0])

(str, int)

In [103]:
m1 = parents_df.merge(df_order_counts, left_on='parent_material', right_on='Material', how='inner')
m1.head()

,parent_material,child_material,Material,MaterialText,order_counts
0,29012383/0130,S091823,29012383/0130,CABLE -120-02-W2,587
1,29012383/0130,S095170,29012383/0130,CABLE -120-02-W2,587
2,29012383/0130,S099116,29012383/0130,CABLE -120-02-W2,587
3,29012383/0133,195248,29012383/0133,CABLE -615-06-05-W2,28
4,29012383/0133,S093197,29012383/0133,CABLE -615-06-05-W2,28


In [104]:
m1.shape

(221, 5)

In [105]:
df_order_counts.columns, parents_df.columns

(Index(['Material', 'MaterialText', 'order_counts'], dtype='str'),
 Index(['parent_material', 'child_material'], dtype='str'))

In [107]:
df_order_counts['Material'].nunique()

67396

In [128]:
m2 = m1.merge(df_order_counts, left_on='child_material', right_on='Material', how='left', suffixes=('_parent', '_child'))
m2.head()

,parent_material,child_material,Material_parent,MaterialText_parent,order_counts_parent,Material_child,MaterialText_child,order_counts_child
0,29012383/0130,S091823,29012383/0130,CABLE -120-02-W2,587,NaN,NaN,NaN
1,29012383/0130,S095170,29012383/0130,CABLE -120-02-W2,587,S095170,SHRINK PRINT YE Ø4.7-9.5 ZH,119.0
2,29012383/0130,S099116,29012383/0130,CABLE -120-02-W2,587,S099116,"HEAT SHRINKABLE TUBE,9.5 mm,2:1,WHITE",75.0
3,29012383/0133,195248,29012383/0133,CABLE -615-06-05-W2,28,NaN,NaN,NaN
4,29012383/0133,S093197,29012383/0133,CABLE -615-06-05-W2,28,S093197,"TERMINAL,LUG,END,0.5 mm2,8 mm",298.0


In [112]:
m2.dropna(subset=['Material_child'], inplace=True)
m2.shape

(173, 8)

In [114]:
m2.drop(columns=['Material_parent','Material_child'], inplace=True)
m2.head()

,parent_material,child_material,MaterialText_parent,order_counts_parent,MaterialText_child,order_counts_child
1,29012383/0130,S095170,CABLE -120-02-W2,587,SHRINK PRINT YE Ø4.7-9.5 ZH,119.0
2,29012383/0130,S099116,CABLE -120-02-W2,587,"HEAT SHRINKABLE TUBE,9.5 mm,2:1,WHITE",75.0
4,29012383/0133,S093197,CABLE -615-06-05-W2,28,"TERMINAL,LUG,END,0.5 mm2,8 mm",298.0
5,29012383/0133,S094667,CABLE -615-06-05-W2,28,SHRINK HEAT 2:1 PO9.5X0.6 BLACK,77.0
6,29012383/0133,S099113,CABLE -615-06-05-W2,28,HEAT SHRINK 3.2mm 2:1 WHITE,16.0


In [115]:
df1 = df[['Serviceeventcode','Itemnumber']]
df1.head()

,Serviceeventcode,Itemnumber
0,BladeBearingReplace,107033
1,BladeBearingReplace,760306
2,BladeBearingReplace,76221907
3,BladeBearingReplace,76221971
4,GenBearReplace,103566


In [116]:
m3 = m2.merge(df1, how='inner', left_on='parent_material', right_on='Itemnumber', suffixes=('_m2', '_df1'))
m3.head()

,parent_material,child_material,MaterialText_parent,order_counts_parent,MaterialText_child,order_counts_child,Serviceeventcode,Itemnumber
0,29012383/0130,S095170,CABLE -120-02-W2,587,SHRINK PRINT YE Ø4.7-9.5 ZH,119.0,HubCANCableReplace,29012383/0130
1,29012383/0130,S099116,CABLE -120-02-W2,587,"HEAT SHRINKABLE TUBE,9.5 mm,2:1,WHITE",75.0,HubCANCableReplace,29012383/0130
2,29012383/0133,S093197,CABLE -615-06-05-W2,28,"TERMINAL,LUG,END,0.5 mm2,8 mm",298.0,CCICableReplace,29012383/0133
3,29012383/0133,S094667,CABLE -615-06-05-W2,28,SHRINK HEAT 2:1 PO9.5X0.6 BLACK,77.0,CCICableReplace,29012383/0133
4,29012383/0133,S099113,CABLE -615-06-05-W2,28,HEAT SHRINK 3.2mm 2:1 WHITE,16.0,CCICableReplace,29012383/0133


In [132]:
# No parent child relationship exists in a same event with order countsss
m3a = m3.merge(df1, left_on=['Serviceeventcode','child_material'], right_on=['Serviceeventcode','Itemnumber'], how='inner')
m3a.head()

,parent_material,child_material,MaterialText_parent,order_counts_parent,MaterialText_child,order_counts_child,Serviceeventcode,Itemnumber_x,Itemnumber_y


In [118]:
m3.shape, m3['Serviceeventcode'].unique()

((173, 8),
 <ArrowStringArray>
 [  'HubCANCableReplace',      'CCICableReplace',     'ConvCableReplace',
  'PitchHydCableReplace', 'BladeLubCableReplace']
 Length: 5, dtype: str)

In [120]:
m3.loc[m3['Serviceeventcode']=='ConvCableReplace']

,parent_material,child_material,MaterialText_parent,order_counts_parent,MaterialText_child,order_counts_child,Serviceeventcode,Itemnumber
22,29012710/0110,S099118,CABLE -405-04-01-F2:2,3,HEAT SHRINK 19.0mm 2:1 WHITE,9.0,ConvCableReplace,29012710/0110
23,29012710/0145,S094414,CABLE -405-04-01-R1:6,29,TERMINAL KRF 16mm2 M8 90DEG,11.0,ConvCableReplace,29012710/0145
24,29012710/0145,S099117,CABLE -405-04-01-R1:6,29,"HEAT SHRINKABLE TUBE,12.7 mm,2:1,WHITE",40.0,ConvCableReplace,29012710/0145
25,79541002/101,S099117,CABLE 405-02-05-Q14-146.1:T3,3,"HEAT SHRINKABLE TUBE,12.7 mm,2:1,WHITE",40.0,ConvCableReplace,79541002/101


In [123]:
req_cols = ['Serviceeventcode','parent_material','order_counts_parent','order_counts_child']
m4 = m3[req_cols]
m4.head()

,Serviceeventcode,parent_material,order_counts_parent,order_counts_child
0,HubCANCableReplace,29012383/0130,587,119.0
1,HubCANCableReplace,29012383/0130,587,75.0
2,CCICableReplace,29012383/0133,28,298.0
3,CCICableReplace,29012383/0133,28,77.0
4,CCICableReplace,29012383/0133,28,16.0


In [129]:
pc = m4.groupby(['Serviceeventcode','parent_material']).agg({'order_counts_parent':'mean', 'order_counts_child':'sum'}).reset_index()
pc['parent_to_child'] = pc['order_counts_parent'] / pc['order_counts_child']
pc.head()

,Serviceeventcode,parent_material,order_counts_parent,order_counts_child,parent_to_child
0,BladeLubCableReplace,29012383/0100,465.0,651.0,0.714286
1,BladeLubCableReplace,29012383/0126,380.0,390.0,0.974359
2,CCICableReplace,29012383/0133,28.0,466.0,0.060086
3,CCICableReplace,29012385/0105,310.0,540.0,0.574074
4,CCICableReplace,29012387/0100,17.0,3.0,5.666667


In [137]:
eves= pc.loc[pc['parent_to_child']>0.5,'Serviceeventcode'].unique()
m3.loc[m3['Serviceeventcode'].isin(eves)]

,parent_material,child_material,MaterialText_parent,order_counts_parent,MaterialText_child,order_counts_child,Serviceeventcode,Itemnumber
0,29012383/0130,S095170,CABLE -120-02-W2,587,SHRINK PRINT YE Ø4.7-9.5 ZH,119.0,HubCANCableReplace,29012383/0130
1,29012383/0130,S099116,CABLE -120-02-W2,587,"HEAT SHRINKABLE TUBE,9.5 mm,2:1,WHITE",75.0,HubCANCableReplace,29012383/0130
2,29012383/0133,S093197,CABLE -615-06-05-W2,28,"TERMINAL,LUG,END,0.5 mm2,8 mm",298.0,CCICableReplace,29012383/0133
3,29012383/0133,S094667,CABLE -615-06-05-W2,28,SHRINK HEAT 2:1 PO9.5X0.6 BLACK,77.0,CCICableReplace,29012383/0133
4,29012383/0133,S099113,CABLE -615-06-05-W2,28,HEAT SHRINK 3.2mm 2:1 WHITE,16.0,CCICableReplace,29012383/0133
5,29012383/0133,S099116,CABLE -615-06-05-W2,28,"HEAT SHRINKABLE TUBE,9.5 mm,2:1,WHITE",75.0,CCICableReplace,29012383/0133
6,29012385/0105,S091884,CABLE -615-06-11-W2 L=7M,310,CABLE CAN M12 MALE L=7M UL,2.0,CCICableReplace,29012385/0105
7,29012385/0105,S094090,CABLE -615-06-11-W2 L=7M,310,PHOENIXPLUG QC0.5/2-ST-3.81 BK,267.0,CCICableReplace,29012385/0105
8,29012385/0105,S094667,CABLE -615-06-11-W2 L=7M,310,SHRINK HEAT 2:1 PO9.5X0.6 BLACK,77.0,CCICableReplace,29012385/0105
9,29012385/0105,S095170,CABLE -615-06-11-W2 L=7M,310,SHRINK PRINT YE Ø4.7-9.5 ZH,119.0,CCICableReplace,29012385/0105
